# PoseMamba: 3D pose from a video (Colab)

Upload a video of a person, get back a video with the 2D skeleton on the footage and a 3D skeleton beside it.

**How to use**
1. Runtime > Change runtime type > **T4 GPU**.
2. Run **Step 1** and **Step 2** once per session (about 15 min, mostly compiling). Run them top to bottom.
3. Run **Step 3** for each video you want to process.

A Colab session resets after a timeout, so Steps 1 and 2 have to be repeated in a new session. Within one session, only Step 3 is needed for new videos.

## Step 1: Setup (libraries, project code, GPU kernels)

In [ ]:
import os, sys, subprocess, glob, torch

assert torch.cuda.is_available(), "No GPU found. Runtime > Change runtime type > T4 GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| CUDA", torch.version.cuda)

if not os.path.isdir("/content/PoseMamba"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/nankingjing/PoseMamba.git", "/content/PoseMamba"], check=True)
os.chdir("/content/PoseMamba")

# Python packages the project needs (keeps the PyTorch that Colab already has)
!pip install -q -r requirements.txt gdown
print("packages ready")

In [ ]:
# Build the two custom GPU kernels (selective scan: 'core' and 'oflex').
# The repo targets old CUDA, so three small calls removed in CUDA 13 are patched,
# and the build list is widened because the repo only builds 'oflex' by default.
KDIR = "/content/PoseMamba/kernels/selective_scan"
os.chdir(KDIR)

# 1) CUDA 13 removed cub::LaneId and cub::CTA_SYNC; use plain CUDA equivalents
!sed -i 's/cub::LaneId()/(threadIdx.x \& 31)/; s/cub::CTA_SYNC()/__syncthreads()/' csrc/selective_scan/reverse_scan.cuh
# 2) build 'core' as well as 'oflex' (the model calls both)
!sed -i 's/^MODES = \["oflex"\]/MODES = ["core", "oflex"]/' setup.py

if len(glob.glob(KDIR + "/*.so")) < 2:
    print("compiling kernels, this takes several minutes...")
    r = subprocess.run("python setup.py build_ext --inplace > /content/build.log 2>&1", shell=True)
    if r.returncode != 0:
        print("BUILD FAILED. Last lines of the build log:")
        !grep -nE " error|Error:" /content/build.log | head -20
        raise SystemExit("Fix the error above, or send it to whoever is helping you.")

print("kernels:", [os.path.basename(p) for p in glob.glob(KDIR + "/*.so")])

# make the compiled kernels importable here and in the demo script
sys.path.insert(0, KDIR)
os.environ["PYTHONPATH"] = KDIR
import selective_scan_cuda_core, selective_scan_cuda_oflex
print("kernels load OK")
os.chdir("/content/PoseMamba")

## Step 2: Download model weights and detector files

In [ ]:
os.chdir("/content/PoseMamba")

# PoseMamba-L weights from the authors' Hugging Face repo
import shutil
from huggingface_hub import hf_hub_download
os.makedirs("checkpoint/pose3d/PoseMamba_L", exist_ok=True)
w = hf_hub_download("nankingjings/PoseMamba-weights", "PoseMamba_L.bin")
shutil.copy(w, "checkpoint/pose3d/PoseMamba_L/best_epoch.bin")
shutil.copy("configs/pose3d/PoseMamba_train_h36m_L.yaml", "checkpoint/pose3d/PoseMamba_L/config.yaml")

# The demo/ folder (2D person detector code + its weights + sample videos, ~1.1 GB) is not in the git repo.
# It comes from the Google Drive file linked in the project README.
if not os.path.isdir("demo/lib/checkpoint"):
    !gdown 1hbK1HDz1nMTGYcczOC5r33Mk8nAtLZCr -O /content/demo.tar
    !tar -xf /content/demo.tar -C /content/PoseMamba

# PyTorch 2.6+ refuses to load older checkpoints unless told to. The weights come from the
# project authors' own repo, so allow it for the one call in vis.py that loads them.
src = open("vis.py").read()
if "weights_only" not in src:
    src = src.replace("map_location=lambda storage, loc: storage)",
                      "map_location=lambda storage, loc: storage, weights_only=False)")
    open("vis.py", "w").write(src)

print("demo files:", os.listdir("demo/video"))
print("setup complete")

## Step 3: Run it on a video

Set `USE_SAMPLE = False` to upload your own video (one person, clear view, a few seconds works best).
Processing time is mostly the final drawing stage, about 1 second per frame.

In [ ]:
USE_SAMPLE = True   # True = bundled sample clip, False = upload your own

os.chdir("/content/PoseMamba")
if USE_SAMPLE:
    video_name = "tj_cut.mp4"
else:
    from google.colab import files
    up = files.upload()
    video_name = list(up.keys())[0]
    shutil.move(video_name, "demo/video/" + video_name)

stem = os.path.splitext(video_name)[0]
!python vis.py --video {video_name} --gpu 0 2>&1 | grep -v -e FutureWarning -e "@torch"

out = f"/content/PoseMamba/demo/output/{stem}/{stem}.mp4"
print("\nOutput video:", out, "| exists:", os.path.exists(out))

In [ ]:
# Download the result to your computer
from google.colab import files
files.download(out)